# Deterministic matching: exact → fuzzy

Resolves duplicate CONT_IDs with rules, in two cascading stages:

1. **exact**: raw equality on key column combinations; conservative.
2. **fuzzy**: normalized and similarity comparisons. A pair is kept only if at least one of its records is still unresolved, so a leftover record can still join a cluster found earlier.

**Entity matching vs entity resolution:**
- **Matching** is pair by pair: a pair is accepted *automatically* when every condition of one rule holds (score 1.0, no review).
- **Resolution** turns the accepted pairs into entities with the configured `Resolver`. `connected_components` is transitive: A–B and B–C make one entity even if A–C never matched. `constrained` refuses a merge whose clusters hold different values in a cannot-link column.
- Metrics are reported at both levels: `pairs_*` on the accepted pairs, the rest on the clusters (B-cubed, over-merged and split entities).

**Everything is config.** Column names appear only in the Configuration cell. The engine is the shared `er` package. Each column's comparison logic (normalization → function → threshold) is one `Comparator` per stage, so an A/B test or the Optuna search only swaps comparators.

**Every search reports its time complexity.** A rule compares only the pairs inside its exact-key blocks: **P = Σ C(k, 2)** over block sizes k. Its cost is `O(N + P)` for exact rules and `O(N + P·c)` for fuzzy ones, where c is the per-pair cost of the string function.

**Datasets** (`data/medallion/silver/<dataset>.csv`, truth in `data/labels/`):
- **train**: design analysis and Optuna. The rules have no fitted parameters, so train is the selection set.
- **val**: A/B tests (paired against `SPEC`) and the choice among Pareto points
- **test**: the final report, once per configuration; the leaderboard counts test evaluations
- **stress / large**: robustness and FP-rate certification, report-only and never tuned on

**Uncertainty everywhere:**
- `recall_lo` / `recall_hi`: 95% Wilson interval
- `fp_rate_hi`: 95% upper bound on the FP rate among predicted pairs; zero FP among n pairs only bounds the rate at ≈ 3/n

**Outputs**, for every dataset:
- `data/artifacts/matched/<dataset>/deterministic_pairs.csv`: matching
- `data/artifacts/resolved/<dataset>/deterministic_resolved.csv` and `data/artifacts/unresolved/<dataset>/deterministic_unresolved.csv`: resolution, and the hand-off

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from er.config import (Budget, Candidate, Comparator, Data, Derived, OverBudget, Resolver, Rule, RulesSpec, Stage,
                       Standardization, Tune, with_comparators)
from er.evaluate import c2, paired_test, predicted_pairs, true_pairs
from er.io import append_leaderboard, load, test_evaluations, write_stage
from er.pipeline import Result, check_budget, complexity_summary, preflight, run_rules
from er.standardize import prepare, self_check
from er.strings import connect

## Configuration
The only cell that names columns. Defaults follow the measured column agreement:
- codes (CIF_ID, PHONE_NUM_4, SOURCE_ADDR_CD, postal) are compared exactly
- names and address text are fuzzy
- CIF_ID never stands alone
- fallback columns are folded into derived columns (`PERSON_NAME`, `POSTAL`, `ADDRESS`)

In [ ]:
DATA = Data(
    silver_dir=Path("../data/medallion/silver"),
    labels_dir=Path("../data/labels"),
    gold_dir=Path("../data/medallion/gold"),
    artifacts_dir=Path("../data/artifacts"),
    experiments_dir=Path("../data/artifacts/experiments"),
    models_dir=Path("../data/artifacts/models"),
    datasets=("full", "train", "val", "test", "stress", "large"),
    robustness=("stress", "large"),
    unique_id="CONT_ID",
    entity_id="ENTITY_ID",
    show_rows=8,
)

STANDARDIZE = Standardization(
    kana_fold=dict(zip("ガギグゲゴザジズゼゾダヂヅデドバビブベボパピプペポヴァィゥェォッャュョヮヵヶヲー",
                       "カキクケコサシスセソタチツテトハヒフヘホハヒフヘホフアイウエオツヤユヨワカケオウ", strict=True)),
    legal_forms=("カブシキガイシャ", "ユウゲンガイシャ", "ゴウドウガイシャ", "(カ)", "(ユ)"),
    prefectures=tuple("北海道 青森県 岩手県 宮城県 秋田県 山形県 福島県 茨城県 栃木県 群馬県 埼玉県 千葉県 東京都 神奈川県 新潟県 富山県 "
                      "石川県 福井県 山梨県 長野県 岐阜県 静岡県 愛知県 三重県 滋賀県 京都府 大阪府 兵庫県 奈良県 和歌山県 鳥取県 島根県 "
                      "岡山県 広島県 山口県 徳島県 香川県 愛媛県 高知県 福岡県 佐賀県 長崎県 熊本県 大分県 宮崎県 鹿児島県 沖縄県".split()),
    address_replacements=((r"(\d+)丁目", r"\1-"), (r"(\d+)番地?", r"\1-"), (r"(\d+)号", r"\1")),
    address_cut_at=" ",
    derived=(
        Derived(name="PERSON_NAME", sources=(("FULL_NAME_KANA",), ("O_PERSON_NAME_KANA",), ("LAST_NAME_KANA", "GIVEN_NAME_KANA")),
                normalize=("strip_spaces",)),
        Derived(name="POSTAL", sources=(("POST_CD",), ("POST_CD_KANA",))),
        Derived(name="ADDRESS", sources=(("STD_ADDRESS",), ("ORIGINAL_ADDRESS",))),
    ),
)

SPEC = RulesSpec(
    standardize=STANDARDIZE,
    stages=(
        Stage(name="exact", rules=(
            Rule(name="X1_CIF_PERSON_NAME", exact=("CIF_ID", "FULL_NAME_KANA")),
            Rule(name="X2_CIF_COMPANY_NAME", exact=("CIF_ID", "NYS_CNV_P_NAME_KANA")),
            Rule(name="X3_CIF_PHONE", exact=("CIF_ID", "PHONE_NUM_4")),
            Rule(name="X4_PERSON_NAME_ADDRCD_PHONE", exact=("FULL_NAME_KANA", "SOURCE_ADDR_CD", "PHONE_NUM_4")),
            Rule(name="X5_COMPANY_NAME_ADDRCD_PHONE", exact=("NYS_CNV_P_NAME_KANA", "SOURCE_ADDR_CD", "PHONE_NUM_4")),
        )),
        Stage(name="fuzzy",
              comparators={
                  "PERSON_NAME": Comparator("jaro_winkler", 0.92, ("kana_fold",)),
                  "NYS_CNV_P_NAME_KANA": Comparator("jaro_winkler", 0.92, ("nfkc", "strip_legal_form", "kana_fold")),
                  "ADDRESS": Comparator("exact", normalize=("address",)),
                  "PHONE_NUM_4": Comparator("hamming", 1),
              },
              rules=(
                  Rule(name="F1_NAME~_ADDRCD_PHONE", exact=("SOURCE_ADDR_CD", "PHONE_NUM_4"), fuzzy=("PERSON_NAME",)),
                  Rule(name="F2_NAME~_ADDRESS~_PHONE", exact=("PHONE_NUM_4",), fuzzy=("PERSON_NAME", "ADDRESS")),
                  Rule(name="F3_NAME~_CIF", exact=("CIF_ID",), fuzzy=("PERSON_NAME",)),
                  Rule(name="F4_NAME_ADDRCD_PHONE~", exact=("PERSON_NAME", "SOURCE_ADDR_CD"), fuzzy=("PHONE_NUM_4",)),
                  Rule(name="F5_COMPANY~_PHONE_POSTAL", exact=("PHONE_NUM_4", "POSTAL"), fuzzy=("NYS_CNV_P_NAME_KANA",)),
                  Rule(name="F6_COMPANY~_ADDRCD_PHONE", exact=("SOURCE_ADDR_CD", "PHONE_NUM_4"), fuzzy=("NYS_CNV_P_NAME_KANA",)),
                  Rule(name="F7_GIVEN_ADDRCD_PHONE", exact=("GIVEN_NAME_KANA", "SOURCE_ADDR_CD", "PHONE_NUM_4")),  # surname change
              )),
    ),
    resolver=Resolver(method="connected_components"),
    budget=Budget(max_candidate_pairs=5_000_000, max_block_size=5_000, target_rows=1_000_000, benchmark_pairs=20_000),
)

VARIANTS: dict[str, RulesSpec] = {
    "A_fold_jaro_winkler": SPEC,
    "B_fold_levenshtein": with_comparators(SPEC, "fuzzy", {
        "PERSON_NAME": Comparator("levenshtein", 1, ("kana_fold",)),
        "NYS_CNV_P_NAME_KANA": Comparator("levenshtein", 1, ("nfkc", "strip_legal_form", "kana_fold"))}),
    "C_fold_exact": with_comparators(SPEC, "fuzzy", {
        "PERSON_NAME": Comparator("exact", normalize=("kana_fold",)),
        "NYS_CNV_P_NAME_KANA": Comparator("exact", normalize=("nfkc", "strip_legal_form", "kana_fold"))}),
    "D_raw_jaro_winkler": with_comparators(SPEC, "fuzzy", {
        "PERSON_NAME": Comparator("jaro_winkler", 0.92),
        "NYS_CNV_P_NAME_KANA": Comparator("jaro_winkler", 0.92)}),
    "E_constrained_given_name": RulesSpec(standardize=SPEC.standardize, stages=SPEC.stages, budget=SPEC.budget,
                                          resolver=Resolver(method="constrained", cannot_link=("GIVEN_NAME_KANA",))),
}

TUNE = Tune(
    stage="fuzzy",
    space={
        "PERSON_NAME": (Candidate("exact", ("kana_fold",)),
                        Candidate("jaro_winkler", ("kana_fold",), 0.85, 0.98, 0.01),
                        Candidate("jaro_winkler", (), 0.85, 0.98, 0.01),
                        Candidate("levenshtein", ("kana_fold",), 1, 2, 1),
                        Candidate("damerau_levenshtein", ("kana_fold",), 1, 2, 1)),
        "NYS_CNV_P_NAME_KANA": (Candidate("exact", ("nfkc", "strip_legal_form", "kana_fold")),
                                Candidate("jaro_winkler", ("nfkc", "strip_legal_form", "kana_fold"), 0.88, 0.98, 0.01),
                                Candidate("levenshtein", ("nfkc", "strip_legal_form", "kana_fold"), 1, 3, 1),
                                Candidate("jaccard", (), 0.75, 0.95, 0.01)),
        "ADDRESS": (Candidate("exact", ("address",)),
                    Candidate("levenshtein", ("address",), 1, 3, 1),
                    Candidate("jaro_winkler", ("address",), 0.90, 0.99, 0.01)),
        "PHONE_NUM_4": (Candidate("hamming", (), 1, 1, 1),
                        Candidate("damerau_levenshtein", (), 1, 1, 1)),
    },
    n_trials=60,
    timeout_s=900,
    seed=0,
    storage_path=Path("../data/artifacts/optuna/deterministic_matching.db"),
    study_prefix="deterministic",
    dashboard_host="127.0.0.1",
    dashboard_port=8080,
)

## Setup
DuckDB with character-level `cl_*` string functions (self-checked), and the address normalization self-check.

In [ ]:
CON, DB = connect()
self_check(STANDARDIZE)
uid = DATA.unique_id


def run(spec: RulesSpec, records: pd.DataFrame, truth: pd.DataFrame) -> Result:
    return run_rules(spec, records, truth, DB, DATA)


def truth_set(truth: pd.DataFrame) -> set[tuple[str, str]]:
    return set(map(tuple, true_pairs(truth.set_index(uid)[DATA.entity_id], uid).to_numpy()))


def joined(res: Result) -> set[tuple[str, str]]:  # every pair the clusters put together
    return predicted_pairs(res.records.set_index(uid)["CLUSTER_ID"])

## Preflight: time complexity before any join
Runs on the largest dataset. For each rule:
- its Big-O with N and P filled in
- its largest block
- an estimated fuzzy time, from timing each function on sample pairs
- P projected to `budget.target_rows` (P grows with N²)

The cell stops if a rule is over budget.

In [ ]:
sizes = {d: len(load(DATA, d)[0]) for d in DATA.datasets}
largest = max(sizes, key=sizes.get)
big = load(DATA, largest)[0]
big_plan = preflight(SPEC, prepare(SPEC.standardize, big, SPEC.normalized(), SPEC.used()))
display(big_plan.drop(columns="ops"))
print(f"{largest}: normalize + derive: O(N·L), N = {len(big):,} · group-by per rule: O(N) · "
      f"total candidate pairs P = {big_plan['P'].sum():,} → {big_plan['projected_P'].sum():,} at {SPEC.budget.target_rows:,} rows")
check_budget(big_plan, SPEC.budget)
print("within budget")

## Design analysis on train only
This analysis justifies the default rules. It runs on **train** only, never on data that contains val or test, so the rule design can't leak the evaluation sets.

For each column and each candidate key combination: the share of same-entity pairs that agree (recall side) vs. the number of different-entity pairs that agree (FP side).

In [ ]:
train, train_truth = load(DATA, "train")
t_df = prepare(SPEC.standardize, train, SPEC.normalized(), SPEC.used()).merge(train_truth[[uid, DATA.entity_id]], on=uid)
n_true = c2(t_df.groupby(DATA.entity_id).size())
rows = []
for keys in [[c] for c in sorted(SPEC.used())] + [list(s.keys(r)) for s in SPEC.stages for r in s.rules]:
    d = t_df.dropna(subset=keys)
    agree, same = c2(d.groupby(keys).size()), c2(d.groupby(keys + [DATA.entity_id]).size())
    rows.append(dict(keys=" + ".join(keys), same_entity_agree=same / n_true, different_entity_agree_pairs=agree - same))
print(f"train: {len(t_df):,} records, {n_true:,} true pairs · cost O(N) per key (a group-by)")
display(pd.DataFrame(rows).drop_duplicates("keys").round(4))

## Evaluation: SPEC on val and test
Stage metrics are cumulative: the fuzzy row includes the exact stage's links.
- `pairs_*` columns: **matching**, the accepted pairs themselves
- other columns: **resolution**, the clusters after closure
- `recall_lo` / `recall_hi` and `fp_rate_hi`: the uncertainty; "0 FP" on n predicted pairs still allows an FP rate up to `fp_rate_hi`

In [ ]:
results = {split: run(SPEC, *load(DATA, split)) for split in ("val", "test")}
display(pd.concat({k: v.metrics for k, v in results.items()}, names=["split"]).round(4))
for split, res in results.items():
    print(f"{split}: {complexity_summary(res)}")
val = results["val"]
display(val.plan[["stage", "rule", "big_o", "filled", "P", "largest_block", "est_fuzzy_ms", "actual_ms"]].merge(val.rules, on="rule"))

In [ ]:
cols = sorted(SPEC.used())
fp = val.edges[val.edges["kept"] & val.edges["fp"]].head(DATA.show_rows)
side = val.records.set_index(uid)[cols]
print(f"val false-positive pairs: {int((val.edges['kept'] & val.edges['fp']).sum())}")
if len(fp):
    display(pd.concat([fp[["rule"]].reset_index(drop=True),
                       side.loc[fp[f"{uid}_l"]].add_suffix("_l").reset_index(drop=True),
                       side.loc[fp[f"{uid}_r"]].add_suffix("_r").reset_index(drop=True)], axis=1).T)

val_truth = load(DATA, "val")[1].set_index(uid)[DATA.entity_id]
left = val.records.assign(entity=val.records[uid].map(val_truth))
left = left[left["CLUSTER_ID"].map(left["CLUSTER_ID"].value_counts()).eq(1) & left["entity"].map(left["entity"].value_counts()).gt(1)]
print(f"val true duplicates still unresolved ({len(left)} records), sample:")
display(left[left["entity"].isin(left["entity"].unique()[:DATA.show_rows // 2])].sort_values("entity")[["entity", uid, *cols]])

## A/B: variants on val, paired against SPEC
Recall intervals of close variants overlap, so each variant is compared with `SPEC` **pair by pair** on the same records:
- `TP_gains` / `TP_losses`: true pairs only the variant joins / only `SPEC` joins
- `p`: an exact sign test on those two counts
- `FP_added` / `FP_removed`: the same for false pairs

Rows are also appended to `data/artifacts/experiments/leaderboard.csv`.

In [ ]:
ab = []
val_records, val_truth_df = load(DATA, "val")
val_true = truth_set(val_truth_df)
base = joined(run(SPEC, val_records, val_truth_df))
for name, spec in VARIANTS.items():
    res = run(spec, val_records, val_truth_df)
    m = res.metrics.iloc[-1]
    row = dict(FP=m["FP"], fp_rate_hi=m["fp_rate_hi"], recall=m["recall"], recall_lo=m["recall_lo"], recall_hi=m["recall_hi"],
               precision=m["precision"], f1=m["f1"], overmerged=m["overmerged"], **paired_test(base, joined(res), val_true),
               P=int(res.plan["P"].sum()), ops=float(res.plan["ops"].sum()),
               big_o=" + ".join(sorted(set(res.plan["big_o"]))), seconds=round(res.seconds, 1))
    append_leaderboard(DATA, f"deterministic:{name}", spec, "val", row)
    ab.append(dict(variant=name, **row))
display(pd.DataFrame(ab).round(4))

## Robustness: SPEC on stress and large (report only, never tuned on)
- **stress:** another seed with more typos (names, phones and addresses), less nesting and more households. A configuration that only fits the generator's defaults shows it here.
- **large:** 50k entities. Its many predicted pairs give a tight FP-rate upper bound.

In [ ]:
robust = {d: run(SPEC, *load(DATA, d)) for d in DATA.robustness}
display(pd.concat({d: r.metrics for d, r in robust.items()}, names=["dataset"])[
    ["stage", "TP", "FP", "recall", "recall_lo", "recall_hi", "fp_rate_hi", "overmerged", "dup_unresolved"]].round(4))
for d, r in robust.items():
    m = r.metrics.iloc[-1]
    print(f"{d}: {complexity_summary(r)} · {int(m['FP'])} FP in {int(m['TP'] + m['FP']):,} predicted pairs → FP rate ≤ {m['fp_rate_hi']:.3%}")

## Output: standard stage files for every dataset
For every dataset, this cell writes:
- the accepted pairs (matching)
- the resolved clusters and the unresolved records (resolution)

The probabilistic notebook reads these. Truth-derived columns are never written into them.

The test counter shows how many distinct configurations have been scored on test. More than one per family means test influenced a choice.

In [ ]:
for dataset in DATA.datasets:
    records, truth = load(DATA, dataset)
    res = run(SPEC, records, truth)
    pairs = res.edges[[f"{uid}_l", f"{uid}_r", "stage", "rule", "score", "kept"]]
    resolved = res.records[[uid, "CLUSTER_ID", "MATCH_STAGE", "MATCH_RULE"]].set_axis(
        [uid, "cluster_id", "resolved_stage", "resolved_by"], axis=1)
    unresolved = res.records.loc[res.records["MATCH_STAGE"].isna(), list(records.columns)]
    assert len(unresolved) + resolved["resolved_stage"].notna().sum() == len(records)
    write_stage(DATA, dataset, "deterministic", pairs, resolved, unresolved)
    append_leaderboard(DATA, "deterministic:SPEC", SPEC, dataset, res.metrics.iloc[-1].drop("stage").to_dict())
    m = res.metrics.iloc[-1]
    print(f"{dataset:>6}: {complexity_summary(res)}\n        recall {m['recall']:.4f} [{m['recall_lo']:.4f}, {m['recall_hi']:.4f}] · "
          f"FP {m['FP']} (rate ≤ {m['fp_rate_hi']:.3%}) · {resolved['resolved_stage'].notna().sum():,} resolved, {len(unresolved):,} unresolved")
counter = test_evaluations(DATA)
display(counter)
for _, row in counter[counter["configs_on_test"] > 1].iterrows():
    print(f"warning: {row['configs_on_test']} configurations of {row['family']!r} were scored on test; its test numbers are no longer unbiased")

## Optuna: search the fuzzy stage's comparators on train
Two objectives: **minimise FP** and **maximise recall**, against train truth. Rules have no fitted parameters, so train is a clean selection set. The result is a Pareto front of trade-offs instead of one winner.

Each trial chooses, per column in `TUNE.space`, a candidate (normalization → function) and then its threshold. Each trial is preflighted: over budget means it is pruned before any join runs.

Front points are re-scored on **val** with intervals. The **suggested** point is the cheapest configuration within the noise of the best val recall at 0 val FP (`pick_within_noise`), which prefers simple over lucky. Test is never used here.

**Live monitoring:** optuna-dashboard at `http://127.0.0.1:8080`. From a terminal in `notebooks/`: `../.venv/bin/optuna-dashboard sqlite:///../data/artifacts/optuna/deterministic_matching.db`.

In [ ]:
import optuna

from er.tune import new_study, pick_within_noise, plain, start_dashboard, suggest_comparators

storage = start_dashboard(TUNE)


def objective(trial: optuna.Trial) -> tuple[float, float]:
    comps = suggest_comparators(trial, TUNE.space)
    trial.set_user_attr("comparators", repr(comps))
    try:
        res = run(with_comparators(SPEC, TUNE.stage, comps), train, train_truth)
    except OverBudget as e:
        raise optuna.TrialPruned(str(e))
    m = res.metrics.iloc[-1]
    trial.set_user_attr("big_o", f"Σ ops ≈ {res.plan['ops'].sum():,.0f} ({' + '.join(sorted(set(res.plan['big_o'])))})")
    for k, v in dict(P=res.plan["P"].sum(), f1=m["f1"], precision=m["precision"], TP=m["TP"], FN=m["FN"],
                     fp_rate_hi=m["fp_rate_hi"], exact_FP=res.metrics.iloc[0]["FP"], seconds=round(res.seconds, 2)).items():
        trial.set_user_attr(k, plain(v))
    return float(m["FP"]), float(m["recall"])


baseline = run(SPEC, train, train_truth)
print(f"study: O(T · per-trial) = O({TUNE.n_trials} · Σ ops ≈ {baseline.plan['ops'].sum():,.0f}); "
      f"per trial ≈ {baseline.seconds:.1f}s → ≈ {TUNE.n_trials * baseline.seconds / 60:.1f} min (timeout {TUNE.timeout_s / 60:.0f} min)")
study = new_study(TUNE, storage, ["FP", "recall"], ["minimize", "maximize"])
study.optimize(objective, n_trials=TUNE.n_trials, timeout=TUNE.timeout_s, show_progress_bar=True)

trials = study.trials_dataframe(attrs=("number", "values", "params", "user_attrs", "state"))
display(trials.rename(columns={"values_FP": "FP", "values_recall": "recall"}).sort_values(["FP", "recall"], ascending=[True, False]).head(20))

In [ ]:
front, seen = [], set()
for t in sorted(study.best_trials, key=lambda t: (t.values[0], -t.values[1])):
    comps = suggest_comparators(optuna.trial.FixedTrial(t.params), TUNE.space)
    if repr(comps) in seen:  # tied trials that picked the same comparators
        continue
    seen.add(repr(comps))
    v = run(with_comparators(SPEC, TUNE.stage, comps), val_records, val_truth_df)
    m = v.metrics.iloc[-1]
    front.append(dict(trial=t.number, train_FP=t.values[0], train_recall=t.values[1], val_FP=m["FP"], val_recall=m["recall"],
                      val_recall_lo=m["recall_lo"], val_recall_hi=m["recall_hi"], val_fp_rate_hi=m["fp_rate_hi"],
                      val_zero_fp=m["FP"] == 0, P=int(v.plan["P"].sum()), big_o=t.user_attrs["big_o"], comparators=comps))
front = pd.DataFrame(front)
suggested = pick_within_noise(front, "val_recall", "val_recall_lo", "val_recall_hi", "val_zero_fp", "P")
print(f"Pareto front: {len(front)} distinct configurations ({len(study.best_trials)} non-dominated trials), train vs val")
display(front.drop(columns="comparators").round(4))
print(f"suggested (cheapest within the noise of the best 0-FP val recall): trial {front.at[suggested, 'trial']}")
print(f"paste-ready comparators for SPEC stage {TUNE.stage!r}:", front.at[suggested, "comparators"])